In [21]:
# Environment setup
# !pip install openai langchain langchain-community faiss-cpu tiktoken dotenv

In [22]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import openai
from dotenv import load_dotenv
import os

# Load .env if you're using one locally (optional)
load_dotenv()

# Set your API key (you can also paste it directly here if needed)
openai.api_key = os.getenv("OPENAI_API_KEY")


# Load high-stakes reference text
with open("high_stakes_reference.txt", "r") as file:
    high_stakes_text = file.read()

# Create the vectorstore using the reference content
embedding_model = OpenAIEmbeddings()
high_stakes_docs = [Document(page_content=high_stakes_text)]
vectorstore = FAISS.from_documents(high_stakes_docs, embedding_model)

def get_high_stakes_score(user_prompt):
    results_with_scores = vectorstore.similarity_search_with_score(user_prompt, k=1)
    matched_text = results_with_scores[0][0].page_content
    score = results_with_scores[0][1]
    return matched_text, score

In [23]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.5)

# Threshold for high-stakes classification (adjust experimentally)
HIGH_STAKES_DISTANCE = 0.5

# Prompt templates
high_stakes_template = PromptTemplate.from_template("""
You are an emergency triage assistant. A user submitted this message:

"{user_prompt}"

Your job is to do three things:
1. Immediately tell them if this seems like a serious or life-threatening situation.
2. Provide concise, actionable guidance (e.g., "See a doctor", "Call 911", "Do not delay").
3. Explain clearly, in under 4 sentences, why this situation may be high risk — using medical or emergency reasoning.

If applicable, include 1–2 reliable sources or links the user could check for further guidance.
""")

low_stakes_template = PromptTemplate.from_template("""
You are a friendly and helpful assistant.

A user asked: "{user_prompt}"

Provide a relevant, informative answer. Keep it engaging and clear.
""")

def route_and_respond(prompt):
    match_text, score = get_high_stakes_score(prompt)
    
    if score <= HIGH_STAKES_DISTANCE:
        stake_label = "🔴 High-Stakes Prompt"
        formatted_prompt = high_stakes_template.format(user_prompt=prompt)
    else:
        stake_label = "⚪ Low-Stakes Prompt"
        formatted_prompt = low_stakes_template.format(user_prompt=prompt)

    print(f"\n========================")
    print(f"User Prompt: {prompt}");
    print(f"{stake_label} (Score: {score:.4f})");

    # Get response from LLM
    response = llm.predict(formatted_prompt)
    print(f"\n🧠 Response:\n{response}");
    return response

In [26]:
# You can edit this list freely
user_prompts = [
    "My vision is blacking out in one eye, what should I do?",
    "I hit my head and now I feel dizzy and nauseous. Should I go to the hospital?",
    "Explain Uno"
]

In [27]:
for prompt in user_prompts:
    route_and_respond(prompt);


User Prompt: My vision is blacking out in one eye, what should I do?
🔴 High-Stakes Prompt (Score: 0.4668)

🧠 Response:
1. This situation may be serious and potentially life-threatening. 
2. Please seek immediate medical attention by calling 911 or going to the nearest emergency room.
3. Sudden vision loss or blacking out in one eye could be a sign of a serious condition such as a retinal detachment, stroke, or acute glaucoma. Prompt medical evaluation is crucial to determine the cause and prevent further complications.

For more information, please visit the American Academy of Ophthalmology's website: https://www.aao.org/eye-health/tips-prevention/sudden-vision-loss

User Prompt: I hit my head and now I feel dizzy and nauseous. Should I go to the hospital?
🔴 High-Stakes Prompt (Score: 0.4279)

🧠 Response:
1. This situation sounds serious and potentially life-threatening.
2. You should go to the hospital immediately.
3. Head injuries can result in serious complications such as bleedin